# Deep Learning & Transfer Learning with Hugging Face
This lab demonstrates how to leverage pre-trained Transformer models for NLP tasks. We will use the `pipeline` API for zero-shot inference, explore tokenization mechanics, and perform fine-tuning (Transfer Learning) on a custom dataset.

**Hardware Check:** Ensure your GPU is turned on (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!pip install -q transformers datasets evaluate accelerate

### Step 1: The Pipeline API (Inference)
The `pipeline` object abstracts away complex inference code. It downloads a pre-trained model and tokenizer, processes input text, and returns predictions.

In [ ]:
from transformers import pipeline

# Load a pre-trained sentiment analysis model
classifier = pipeline("sentiment-analysis")

results = classifier(["I absolutely love the architecture of this library!", 
                      "The training loop crashed due to out-of-memory errors."])

print("Predictions:")
for result in results:
    print(f"Label: {result['label']}, Confidence: {result['score']:.4f}")

### Step 2: Under the Hood - Tokenization
Neural Networks require numerical matrices as input. The Tokenizer maps textual strings to integer identifiers based on a fixed vocabulary.

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "Machine Learning relies on tensor multiplications."
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"Original Text: {text}")
print(f"Tokens (Sub-words): {tokens}")
print(f"Token IDs (Math): {token_ids}")

### Step 3: Dataset Exploration
Before training, we must understand our data. We will load the 'emotion' dataset.

In [ ]:
from datasets import load_dataset

# Load a tiny subset for demonstration speed
dataset = load_dataset("dair-ai/emotion", split="train[:500]")
dataset = dataset.train_test_split(test_size=0.2)

print("Dataset Structure:")
print(dataset)

print("\nSample Data Point:")
print(dataset['train'][0])

### Step 4: Fine-Tuning (Transfer Learning)
We will download a pre-trained language model that understands English grammar, and *fine-tune* its final classification layer to predict our specific 6 emotion classes.

**Hyperparameters Explained:**
- `learning_rate`: How large of a step the model takes when adjusting its weights. Too high = instability. Too low = slow training.
- `batch_size`: How many sentences the GPU processes at once. Higher batch size utilizes GPU better but requires more memory (VRAM).
- `epochs`: How many times the model sees the entire dataset.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

# 1. Tokenize the entire dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 2. Load the untrained classification model (6 labels for emotions)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

# 3. Setup Training Rules (Hyperparameters)
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,          # Standard rate for fine-tuning Transformers
    per_device_train_batch_size=8, # Process 8 sentences at a time
    num_train_epochs=1,          # Train for 1 cycle (for lab speed)
    weight_decay=0.01,           # Prevents overfitting
)

# 4. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

print("Starting GPU-accelerated training...\n")
trainer.train()
print("\nFine-Tuning Complete! Weights have been updated.")

### Step 5: Post-Training Inference
Let's test our newly fine-tuned model to see if it learned the emotion classes.

In [ ]:
# Move model to CPU for simple inference
model.eval()
model.to('cpu')

test_text = "I am so frustrated that my code is not compiling!"
inputs = tokenizer(test_text, return_tensors="pt")

with torch.no_grad():
    logits = model(**inputs).logits

predicted_class_id = logits.argmax().item()
# 0: sadness, 1: joy, 2: love, 3: anger, 4: fear, 5: surprise
labels = ["Sadness", "Joy", "Love", "Anger", "Fear", "Surprise"]

print(f"Text: {test_text}")
print(f"Predicted Emotion: {labels[predicted_class_id]}")